# 03. 입력 변형 민감도 (탐색 실험)
02의 결과에서 "같은 위치 안" 기준을 통과한 우리 모델이 없었고, 타점 정답으로 학습한 상한 모델(UB_logreg)도 위치 안 AUROC가 0.5 아래였다.
원시 요약값에는 위치 정보가 섞여 있어 모델이 위치 효과를 배우고, 위치 안에서는 그 방향이 뒤집히기 때문이다(01 노트북 2절).
설비·품목·두께가 고정된 KAMP에서는 위치 효과가 없으므로, **위치별 평균을 뺀 입력**(`*_locstd`)이 KAMP 조건에 더 가깝다.

통과 기준은 02와 **같은 기준을 그대로** 적용한다(기준 변경 없음). 이 노트북은 탐색용이며, 최종 입력 조건을 고르는 근거로 쓰려면 새 시드로 확인해야 한다.
- `kamp_full`: 02와 동일(가압력 포함, 위치 미보정) — 비교선
- `kamp_noforce`: 가압력 제외
- `kamp_full_locstd` / `kamp_noforce_locstd`: 위치 평균 제거

In [1]:
import sys, time, numpy as np, pandas as pd, torch
sys.path.insert(0, ".")
from joblib import Parallel, delayed
from wl import runner, bags
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50)

MODELS = ["const_rate", "iforest", "autoencoder", "A_dllp_mlp", "B_count_mlp", "C_prior_two", "C_prior_two_only",
          "D_pseudo_logreg", "UB_logreg", "UB_catboost"]
OURS = ["A_dllp_mlp", "B_count_mlp", "C_prior_two", "C_prior_two_only", "D_pseudo_logreg"]
CONDS = list(bags.CONDITIONS)
FSETS = ["kamp_noforce", "kamp_full_locstd", "kamp_noforce_locstd"]
SEEDS = list(range(5))
N_GPU = torch.cuda.device_count()
RERUN = False

In [2]:
def _task(i, m, c, s, fs):
    return runner.run_one(m, c, s, feature_set=fs, gpu=i % max(N_GPU, 1))

name = "feature_ablation"
if (runner.RES / f"{name}.pkl").exists() and not RERUN:
    runs = runner.load_runs(name)
else:
    tasks = [(m, c, s, fs) for fs in FSETS for m in MODELS for c in CONDS for s in SEEDS]
    t = time.time()
    runs = Parallel(n_jobs=8, verbose=2)(delayed(_task)(i, *k) for i, k in enumerate(tasks))
    print(f"{len(runs)}개 실행, {time.time()-t:.0f}초")
    runner.save_runs(runs, name)
by = {}
for r in runs: by.setdefault((r["feature_set"], r["model"], r["cond"]), []).append(r)

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:    4.0s


C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[Parallel(n_jobs=8)]: Done 146 tasks      | elapsed:  4.6min


[Parallel(n_jobs=8)]: Done 349 tasks      | elapsed:  9.5min


600개 실행, 946초


[Parallel(n_jobs=8)]: Done 600 out of 600 | elapsed: 15.8min finished


In [3]:
def _boot(key): return key, runner.bootstrap(by[key], n_boot=200)
boots = dict(Parallel(n_jobs=16)(delayed(_boot)(k) for k in by))

def fmt(fs, m, c, k, d=3):
    r = boots[(fs, m, c)].loc[k]
    return f"{r['mean']:.{d}f} [{r['lo']:.{d}f}, {r['hi']:.{d}f}]"

KEYS = ["mae", "auroc", "wl_auroc", "recall@0.1", "wl_recall@0.1"]
for fs in FSETS:
    for c in ("orig", "k8_1"):
        print(f"\n=== {fs} | {c} ==="); display(pd.DataFrame({k: {m: fmt(fs, m, c, k) for m in MODELS} for k in KEYS}))


=== kamp_noforce | orig ===


,mae,auroc,wl_auroc,recall@0.1,wl_recall@0.1
const_rate,"2.328 [1.979, 4.801]","0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]"
iforest,"2.328 [1.979, 4.801]","0.601 [0.557, 0.660]","0.454 [0.381, 0.512]","0.129 [0.059, 0.175]","0.075 [0.048, 0.135]"
autoencoder,"2.328 [1.979, 4.801]","0.603 [0.553, 0.647]","0.525 [0.465, 0.595]","0.187 [0.105, 0.236]","0.116 [0.076, 0.147]"
A_dllp_mlp,"2.230 [1.901, 4.641]","0.531 [0.426, 0.603]","0.545 [0.459, 0.607]","0.210 [0.098, 0.299]","0.125 [0.087, 0.163]"
B_count_mlp,"2.832 [2.062, 5.870]","0.624 [0.541, 0.691]","0.441 [0.299, 0.521]","0.213 [0.113, 0.300]","0.089 [0.059, 0.133]"
C_prior_two,"2.474 [2.090, 4.973]","0.585 [0.495, 0.653]","0.446 [0.316, 0.533]","0.147 [0.046, 0.214]","0.084 [0.056, 0.120]"
C_prior_two_only,"2.347 [2.019, 4.729]","0.531 [0.474, 0.569]","0.398 [0.298, 0.470]","0.091 [0.029, 0.146]","0.066 [0.043, 0.098]"
D_pseudo_logreg,"2.319 [1.954, 4.658]","0.697 [0.639, 0.744]","0.400 [0.280, 0.498]","0.170 [0.103, 0.246]","0.092 [0.057, 0.128]"
UB_logreg,"2.367 [1.986, 4.600]","0.740 [0.706, 0.778]","0.294 [0.202, 0.366]","0.276 [0.185, 0.360]","0.036 [0.021, 0.065]"
UB_catboost,"2.147 [1.788, 4.277]","0.809 [0.763, 0.849]","0.435 [0.355, 0.516]","0.528 [0.450, 0.608]","0.092 [0.060, 0.119]"



=== kamp_noforce | k8_1 ===


,mae,auroc,wl_auroc,recall@0.1,wl_recall@0.1
const_rate,"1.442 [1.201, 2.488]","0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]"
iforest,"1.442 [1.201, 2.488]","0.626 [0.567, 0.692]","0.420 [0.316, 0.511]","0.137 [0.037, 0.205]","0.049 [0.017, 0.118]"
autoencoder,"1.442 [1.201, 2.488]","0.630 [0.574, 0.679]","0.543 [0.464, 0.588]","0.179 [0.082, 0.277]","0.102 [0.064, 0.157]"
A_dllp_mlp,"1.470 [1.213, 2.527]","0.618 [0.568, 0.651]","0.394 [0.329, 0.450]","0.111 [0.056, 0.182]","0.056 [0.028, 0.088]"
B_count_mlp,"2.043 [1.526, 4.088]","0.529 [0.472, 0.588]","0.459 [0.360, 0.531]","0.084 [0.032, 0.137]","0.059 [0.039, 0.110]"
C_prior_two,"1.439 [1.271, 2.599]","0.433 [0.390, 0.489]","0.379 [0.308, 0.454]","0.042 [0.000, 0.104]","0.064 [0.026, 0.105]"
C_prior_two_only,"1.389 [1.154, 2.470]","0.480 [0.443, 0.529]","0.343 [0.270, 0.415]","0.076 [0.008, 0.149]","0.048 [0.017, 0.085]"
D_pseudo_logreg,"1.400 [1.170, 2.507]","0.651 [0.601, 0.698]","0.287 [0.205, 0.353]","0.158 [0.093, 0.241]","0.066 [0.026, 0.100]"
UB_logreg,"1.405 [1.223, 2.458]","0.736 [0.686, 0.787]","0.222 [0.147, 0.298]","0.284 [0.168, 0.373]","0.022 [0.000, 0.062]"
UB_catboost,"1.442 [1.104, 2.508]","0.790 [0.730, 0.835]","0.365 [0.275, 0.466]","0.495 [0.360, 0.589]","0.060 [0.032, 0.097]"



=== kamp_full_locstd | orig ===


,mae,auroc,wl_auroc,recall@0.1,wl_recall@0.1
const_rate,"2.328 [1.979, 4.801]","0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]"
iforest,"2.328 [1.979, 4.801]","0.503 [0.453, 0.553]","0.465 [0.410, 0.537]","0.134 [0.086, 0.191]","0.074 [0.052, 0.108]"
autoencoder,"2.328 [1.979, 4.801]","0.562 [0.508, 0.608]","0.522 [0.433, 0.598]","0.175 [0.131, 0.217]","0.083 [0.061, 0.120]"
A_dllp_mlp,"2.103 [1.859, 4.199]","0.581 [0.509, 0.637]","0.501 [0.401, 0.579]","0.181 [0.115, 0.228]","0.089 [0.064, 0.125]"
B_count_mlp,"2.130 [1.744, 4.505]","0.564 [0.499, 0.630]","0.542 [0.433, 0.639]","0.157 [0.077, 0.233]","0.117 [0.073, 0.167]"
C_prior_two,"2.193 [1.878, 4.438]","0.548 [0.492, 0.604]","0.478 [0.398, 0.550]","0.192 [0.111, 0.248]","0.077 [0.053, 0.107]"
C_prior_two_only,"2.328 [1.997, 4.780]","0.432 [0.371, 0.491]","0.404 [0.314, 0.482]","0.144 [0.092, 0.192]","0.078 [0.055, 0.110]"
D_pseudo_logreg,"2.233 [1.779, 4.593]","0.526 [0.439, 0.606]","0.449 [0.341, 0.559]","0.108 [0.030, 0.199]","0.058 [0.041, 0.094]"
UB_logreg,"2.400 [2.095, 4.883]","0.557 [0.461, 0.648]","0.548 [0.432, 0.663]","0.136 [0.073, 0.204]","0.119 [0.082, 0.155]"
UB_catboost,"1.982 [1.527, 4.197]","0.708 [0.656, 0.790]","0.543 [0.444, 0.664]","0.451 [0.356, 0.576]","0.116 [0.074, 0.169]"


... (2 outputs left out)


=== kamp_noforce_locstd | orig ===


,mae,auroc,wl_auroc,recall@0.1,wl_recall@0.1
const_rate,"2.328 [1.979, 4.801]","0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]"
iforest,"2.328 [1.979, 4.801]","0.522 [0.459, 0.563]","0.490 [0.423, 0.565]","0.129 [0.076, 0.184]","0.068 [0.054, 0.112]"
autoencoder,"2.328 [1.979, 4.801]","0.534 [0.490, 0.571]","0.492 [0.423, 0.550]","0.159 [0.120, 0.209]","0.092 [0.061, 0.130]"
A_dllp_mlp,"2.453 [2.105, 4.835]","0.603 [0.530, 0.671]","0.496 [0.416, 0.560]","0.223 [0.132, 0.281]","0.095 [0.074, 0.131]"
B_count_mlp,"2.522 [2.145, 5.131]","0.558 [0.489, 0.616]","0.520 [0.420, 0.595]","0.181 [0.101, 0.243]","0.134 [0.089, 0.162]"
C_prior_two,"2.500 [2.158, 4.907]","0.547 [0.485, 0.602]","0.484 [0.391, 0.574]","0.175 [0.093, 0.232]","0.098 [0.070, 0.134]"
C_prior_two_only,"2.362 [2.023, 4.757]","0.433 [0.372, 0.488]","0.430 [0.353, 0.513]","0.127 [0.072, 0.194]","0.079 [0.062, 0.119]"
D_pseudo_logreg,"2.388 [1.996, 4.802]","0.519 [0.434, 0.584]","0.438 [0.317, 0.523]","0.108 [0.040, 0.172]","0.063 [0.046, 0.096]"
UB_logreg,"2.325 [1.966, 4.792]","0.530 [0.432, 0.620]","0.550 [0.434, 0.673]","0.144 [0.086, 0.215]","0.120 [0.081, 0.157]"
UB_catboost,"2.081 [1.710, 4.188]","0.696 [0.646, 0.762]","0.481 [0.400, 0.561]","0.429 [0.361, 0.532]","0.126 [0.085, 0.153]"



=== kamp_noforce_locstd | k8_1 ===


,mae,auroc,wl_auroc,recall@0.1,wl_recall@0.1
const_rate,"1.442 [1.201, 2.488]","0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]"
iforest,"1.442 [1.201, 2.488]","0.486 [0.422, 0.550]","0.528 [0.448, 0.625]","0.116 [0.047, 0.186]","0.105 [0.066, 0.156]"
autoencoder,"1.442 [1.201, 2.488]","0.530 [0.457, 0.592]","0.543 [0.451, 0.651]","0.158 [0.097, 0.237]","0.104 [0.060, 0.169]"
A_dllp_mlp,"1.434 [1.243, 2.524]","0.508 [0.443, 0.573]","0.470 [0.347, 0.567]","0.137 [0.052, 0.198]","0.114 [0.066, 0.156]"
B_count_mlp,"2.550 [1.864, 4.956]","0.519 [0.460, 0.571]","0.509 [0.420, 0.588]","0.084 [0.027, 0.143]","0.091 [0.065, 0.150]"
C_prior_two,"2.235 [1.552, 4.074]","0.503 [0.441, 0.564]","0.492 [0.412, 0.587]","0.137 [0.054, 0.209]","0.127 [0.066, 0.156]"
C_prior_two_only,"1.416 [1.284, 2.492]","0.400 [0.327, 0.476]","0.420 [0.346, 0.495]","0.105 [0.040, 0.186]","0.101 [0.052, 0.137]"
D_pseudo_logreg,"1.344 [1.194, 2.438]","0.405 [0.365, 0.475]","0.361 [0.256, 0.451]","0.084 [0.034, 0.161]","0.073 [0.034, 0.111]"
UB_logreg,"1.387 [1.247, 2.478]","0.550 [0.457, 0.618]","0.552 [0.449, 0.642]","0.158 [0.080, 0.264]","0.134 [0.079, 0.170]"
UB_catboost,"1.404 [1.079, 2.451]","0.630 [0.554, 0.690]","0.456 [0.345, 0.544]","0.305 [0.224, 0.384]","0.071 [0.044, 0.107]"


## 통과 기준 판정 (02와 동일 기준)
표기: [기준1 MAE][기준2 포착률][기준3 위치안], O=충족 X=미충족. 기준4는 `k8_1`·`k8_05` 둘 다에서 1~3 충족.

In [4]:
def verdict(fs, m, c):
    b = boots[(fs, m, c)]; bc = boots[(fs, "const_rate", c)]
    return (b.loc["mae", "mean"] < bc.loc["mae", "mean"], b.loc["recall@0.1", "lo"] > 0.10,
            b.loc["wl_recall@0.1", "lo"] > 0.10 and b.loc["wl_auroc", "lo"] > 0.5)

rows = []
for fs in FSETS:
    for m in OURS:
        row = {"입력": fs, "모델": m}
        for c in CONDS:
            row[c] = "".join("O" if v else "X" for v in verdict(fs, m, c))
        row["기준4"] = "통과" if all(all(verdict(fs, m, c)) for c in ("k8_1", "k8_05")) else "미통과"
        rows.append(row)
pd.DataFrame(rows)

,입력,모델,orig,k8_4,k8_1,k8_05,기준4
0,kamp_noforce,A_dllp_mlp,OXX,XXX,XXX,XXX,미통과
1,kamp_noforce,B_count_mlp,XOX,OXX,XXX,XXX,미통과
2,kamp_noforce,C_prior_two,XXX,XXX,OXX,XXX,미통과
3,kamp_noforce,C_prior_two_only,XXX,XXX,OXX,OXX,미통과
4,kamp_noforce,D_pseudo_logreg,OOX,XOX,OXX,OXX,미통과
5,kamp_full_locstd,A_dllp_mlp,OOX,XXX,OXX,XXX,미통과
6,kamp_full_locstd,B_count_mlp,OXX,XXX,XXX,XXX,미통과
7,kamp_full_locstd,C_prior_two,OOX,XXX,XXX,XXX,미통과
8,kamp_full_locstd,C_prior_two_only,XXX,OXX,OXX,XXX,미통과
9,kamp_full_locstd,D_pseudo_logreg,OXX,XXX,OXX,OXX,미통과
